In [ ]:
!pip install -q catboost polars lightgbm kaggle

import polars as pl
import pandas as pd
import numpy as np
from datetime import date
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
from catboost import CatBoostClassifier
import os
import warnings
warnings.filterwarnings("ignore")

from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle")

from polars_transformers import (
    Flagnan3Polars, RareLabelPolars, Flagout2Polars,
    DropConstantFeaturesPolars, OrdinalEncPolars, categorizar_todo_polars,
)

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(-1)
print(f"LightGBM {lgb.__version__} | CatBoost {__import__('catboost').__version__}")

LightGBM 4.6.0 | CatBoost 1.2.10


# Santander Product Recommendation — LightGBM and CatBoost using raw vs percentile probability for predicting ranking

## What this notebook does

It is a complementary analysis of the repository related to the same problem that was solved using Lightfm. Here we trains 8 per-product classifiers (LightGBM and CatBoost), validates April → May, and produces May → June submissions. Feature engineering is shared between both model families. Only the algorithm differs.

## Three lessons this notebook documents

**1. Raw probability beats percentile on Kaggle, even though validation cannot distinguish them.**

CatBoost validation:
- Percentile: 0.0234
- Raw: 0.0236
- Validation delta: +0.0002 (noise)

CatBoost leaderboard:
- Percentile: 0.01784
- Raw: 0.02348
- Leaderboard delta: +0.0064

Validation is inflated by train-user overlap and cannot tell them apart. The leaderboard can.

**2. Percentile destroys cross-product signal.**

16 of 24 products are popularity fallbacks (constant across users). Under `rank(pct=True)`:
- Modeled products: percentile varies per user (real signal).
- Fallback products: percentile is the same (~0.5) for every user.

In raw, fallback probability is ~0.0005 and never enters the top-7. In percentile, fallback sits at ~0.5 and competes head-to-head with weakly modeled products. It leaks into the top-7 of users who will not acquire it.

**3. One pipeline for everything.**

The classic LightGBM submission bug (the one that scored poorly) is having a separate `pipe_lgb ≠ pipe_april` or casting to `category` with mismatched categories, introducing `NaN` in May's categoricals. This notebook uses a single `pipe_april` for training, validation, and submission.

In [ ]:
!kaggle competitions download -c santander-product-recommendation
!unzip -o santander-product-recommendation.zip
!unzip -o train_ver2.csv.zip
!unzip -o test_ver2.csv.zip
!unzip -o sample_submission.csv.zip

100% 228M/228M [00:02<00:00, 96.8MB/s]

Archive:  santander-product-recommendation.zip
  inflating: sample_submission.csv.zip  
  inflating: test_ver2.csv.zip       
  inflating: train_ver2.csv.zip      
Archive:  train_ver2.csv.zip
  inflating: train_ver2.csv          
Archive:  test_ver2.csv.zip
  inflating: test_ver2.csv           
Archive:  sample_submission.csv.zip
  inflating: sample_submission.csv   


In [ ]:
DATA_PATH = "/content/train_ver2.csv"
TEST_PATH = "/content/test_ver2.csv"
SAMPLE_SUB = "/content/sample_submission.csv"

NULL_LIKE = [
    "", " ", "NA", "N/A", "NaN", "nan", "null", "NULL",
    "None", "undefined", "NA\x00",
]

# Read every column as text so the scan never fails on type inference
raw_lf = pl.scan_csv(
    DATA_PATH,
    null_values=NULL_LIKE,
    infer_schema_length=0,
)

# 10% deterministic hash sample for profiling (not head, which would concentrate
# the sample in early months because the file is sorted by date)
sample_df = (
    raw_lf
    .filter(pl.col("ncodpers").hash() % 10 == 0)
    .collect()
)

print(f"Sample: {len(sample_df):,} rows, {len(sample_df.columns)} columns")
print(f"Months covered: {sample_df['fecha_dato'].n_unique()}")

Sample: 1,358,466 rows, 48 columns
Months covered: 17


## Data loading

The raw CSV mixes numeric, categorical, date, and genuinely mixed columns. Reading with `infer_schema_length=0` forces every column to string, which guarantees the loader never fails mid-file on a stray label. The diagnostic that follows reconstructs the real types from the data.

This is the most important architectural decision in the pipeline. Every downstream failure in the iteration history traces back to a wrong type inference or an ID leak introduced by treating an identifier as a feature. Getting the schema right from the start prevents those problems.

Three files from the Kaggle competition:

- `train_ver2.csv` — 13.6M rows, 17 months from January 2015 to May 2016
- `test_ver2.csv` — 929,615 rows, one per customer to score for June 2016
- `sample_submission.csv` — the expected format (ncodpers, then space-separated product list)

## Schema diagnostic

The diagnostic reports facts about each column. It does not decide what to do with them. For every column it records missingness, cardinality, parseability as int and float, fractional content, non-numeric labels, and date-pattern match rate.

The classification into types happens *after*, as a rule applied to these facts. Keeping observation and decision separate prevents silent data loss. A column that is 88% numeric and 12% labels will show `non_numeric_pct = 12` rather than being quietly classified as numeric and losing its labels on cast.

Three columns in this dataset carry irregularities that a naive cast would destroy:

- `age` and `antiguedad` contain literal `"NA"` strings that arrive padded with whitespace and are not caught by the CSV `null_values`. These should be null, not zero.
- `indrel_1mes` mixes numeric values `1, 2, 3, 4` with their decimal variants and a small number of `P` labels representing potential customers (~874 rows in the full dataset).
- `canal_entrada` is genuinely mixed: 99.6% alphanumeric channel codes, 0.4% legacy numeric codes.

The first two encode automatically. The third stays as text.

In [ ]:

DATE_FORMATS = [
    (r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"),
    (r"^\d{4}/\d{2}/\d{2}$", "%Y/%m/%d"),
    (r"^\d{2}-\d{2}-\d{4}$", "%d-%m-%Y"),
    (r"^\d{2}/\d{2}/\d{4}$", "%d/%m/%Y"),
]

NULL_LIKE = {"NA", "N/A", "NAN", "NULL", "NONE", "UNDEFINED", "", "UNKNOWN"}

HIGH_NULL_THRESHOLD = 99.0
ZERO_VARIANCE_MAX = 1
DATE_MATCH_MIN = 95.0
TEXT_PURITY_MIN = 95.0
SMALL_LABELS_MAX = 10
def diagnose_polars(sample_df: pl.DataFrame) -> pl.DataFrame:
    total = len(sample_df)
    rows = []

    for col in sample_df.columns:
        s = sample_df[col]
        nulls = s.null_count()
        non_null = total - nulls

        if non_null == 0:
            rows.append({
                "column": col, "null_pct": 100.0, "n_unique": 0,
                "parses_int": False, "parses_float": False,
                "non_numeric_pct": 0.0, "n_dirty_distinct": 0,
                "date_match_pct": 0.0, "has_decimals": False,
                "date_format": None, "dirty_samples": [],
                "min": None, "max": None,
            })
            continue

        cleaned = s.drop_nulls().str.strip_chars()
        n_unique = cleaned.n_unique()

        as_float = cleaned.cast(pl.Float64, strict=False)
        num_vals = as_float.drop_nulls()
        n_num_ok = len(num_vals)

        has_decimals = False
        is_int_ok = False
        if n_num_ok > 0:
            has_decimals = bool((num_vals % 1 != 0).any())
            # parses_int only when every value parses and none carries a fractional part
            if n_num_ok == non_null:
                is_int_ok = not has_decimals

        n_non_num = non_null - n_num_ok
        dirty = cleaned.filter(as_float.is_null())
        n_dirty_distinct = dirty.n_unique()
        dirty_samples = dirty.unique().head(10).to_list()

        best_pct, best_fmt = 0.0, None
        for pattern, fmt in DATE_FORMATS:
            matched = cleaned.str.contains(pattern)
            pct = (matched.sum() / non_null) * 100
            if pct > best_pct:
                best_pct = pct
                best_fmt = fmt

        rows.append({
            "column": col,
            "null_pct": round(nulls / total * 100, 2),
            "n_unique": n_unique,
            "parses_int": is_int_ok,
            "parses_float": bool(n_num_ok == non_null),
            "non_numeric_pct": round(n_non_num / non_null * 100, 2),
            "n_dirty_distinct": n_dirty_distinct,
            "date_match_pct": round(best_pct, 2),
            "date_format": best_fmt if best_pct > 0 else None,
            "has_decimals": has_decimals,
            "dirty_samples": dirty_samples,
            "min": float(num_vals.min()) if len(num_vals) > 0 else None,
            "max": float(num_vals.max()) if len(num_vals) > 0 else None,
        })

    return pl.DataFrame(rows)


def pick_int_dtype_polars(lo, hi):
    if lo is None or hi is None:
        return pl.Int64
    if lo >= 0:
        if hi <= 200: return pl.UInt8
        if hi <= 60_000: return pl.UInt16
        if hi <= 4_000_000_000: return pl.UInt32
    if -100 <= lo and hi <= 100: return pl.Int8
    if -30_000 <= lo and hi <= 30_000: return pl.Int16
    if -2_000_000_000 <= lo and hi <= 2_000_000_000: return pl.Int32
    return pl.Int64




def classify_polars(row: dict) -> str:
    if row["null_pct"] >= HIGH_NULL_THRESHOLD:
        return "drop_high_null"
    if row["n_unique"] <= ZERO_VARIANCE_MAX:
        return "drop_constant"
    if row["date_match_pct"] >= DATE_MATCH_MIN:
        return "date"
    if row["parses_int"]:
        # 0/1 columns are flags; anything else is a plain integer
        if row["n_unique"] == 2 and row["min"] == 0 and row["max"] == 1:
            return "flag"
        return "int"
    if row["parses_float"]:
        return "float"
    if row["non_numeric_pct"] >= TEXT_PURITY_MIN:
        return "text"
    # Single non-numeric label that looks like a null placeholder
    if row["n_dirty_distinct"] == 1 and str(row["dirty_samples"][0]).strip().upper() in NULL_LIKE:
        return "numeric_with_null_label"
    # Mixed numeric + a few labels: safe to encode only when numeric part is integral
    if row["n_dirty_distinct"] <= SMALL_LABELS_MAX:
        if row["has_decimals"] or (row["max"] is not None and row["max"] > 1000):
            # Fractional or very large: encoding would collide with the label codes
            return "numeric_with_null_label"
        return "mixed_small_labels_int"
    return "text"


In [ ]:

def build_curation_plan_polars(diag: pl.DataFrame, sample_df: pl.DataFrame) -> dict:
    plan = {"drop": [], "transforms": {}, "mixed_labels": {}}

    for row in diag.iter_rows(named=True):
        kind = classify_polars(row)
        col = row["column"]

        if kind in ("drop_high_null", "drop_constant"):
            plan["drop"].append(col)

        elif kind == "date":
            plan["transforms"][col] = ("date", row["date_format"])

        elif kind == "flag":
            plan["transforms"][col] = ("flag", pl.UInt8)

        elif kind == "int":
            dtype = pick_int_dtype_polars(row["min"], row["max"])
            plan["transforms"][col] = ("int", dtype)

        elif kind == "float":
            plan["transforms"][col] = ("float", pl.Float64)

        elif kind == "numeric_with_null_label":
            # Preserve integer dtype when the numeric part has no decimals (e.g. age)
            if row["has_decimals"]:
                plan["transforms"][col] = ("numeric_with_null_label", pl.Float64)
            else:
                dtype = pick_int_dtype_polars(row["min"], row["max"])
                plan["transforms"][col] = ("numeric_with_null_label", dtype)

        elif kind == "text":
            plan["transforms"][col] = ("text", pl.String)

        elif kind == "mixed_small_labels_int":
            s_clean = sample_df[col].drop_nulls().str.strip_chars()
            as_float = s_clean.cast(pl.Float64, strict=False)
            labels = s_clean.filter(as_float.is_null()).unique().to_list()
            num_series = as_float.drop_nulls()
            max_num = float(num_series.max()) if len(num_series) > 0 else 0.0
            mapping = {lbl: int(max_num) + i + 1 for i, lbl in enumerate(labels)}
            plan["transforms"][col] = ("mixed_small_labels_int", mapping)
            plan["mixed_labels"][col] = mapping

    return plan


def print_plan(plan: dict) -> None:
    by_kind = {}
    for col, (kind, _meta) in plan["transforms"].items():
        by_kind.setdefault(kind, []).append(col)

    print(f"dropped ({len(plan['drop'])}): {plan['drop']}")
    for kind in ("date", "flag", "int", "float", "numeric_with_null_label",
                 "text", "mixed_small_labels_int"):
        cols = by_kind.get(kind, [])
        if cols:
            print(f"{kind:<25} ({len(cols)}): {cols}")
    for col, mapping in plan["mixed_labels"].items():
        print(f"mixed mapping {col}: {mapping}")

def apply_curation_polars(lf: pl.LazyFrame, plan: dict) -> pl.LazyFrame:
    lf = lf.drop(plan["drop"])
    exprs = []

    for col, (kind, meta) in plan["transforms"].items():
        if kind == "date":
            exprs.append(
                pl.col(col).str.strip_chars().str.to_date(meta, strict=False).alias(col)
            )

        elif kind == "flag":
            exprs.append(
                pl.col(col).str.strip_chars()
                .cast(pl.UInt8, strict=False)
                .fill_null(0)
                .alias(col)
            )

        elif kind == "int":
            exprs.append(
                pl.col(col).str.strip_chars().cast(meta, strict=False).alias(col)
            )

        elif kind == "float":
            exprs.append(
                pl.col(col).str.strip_chars().cast(pl.Float64, strict=False).alias(col)
            )

        elif kind == "numeric_with_null_label":
            stripped = pl.col(col).str.strip_chars()
            if meta == pl.Float64:
                exprs.append(stripped.cast(pl.Float64, strict=False).alias(col))
            else:
                # Strip trailing .0 first so "32.0" casts cleanly to an integer dtype
                exprs.append(
                    stripped.str.replace(r"\.0$", "")
                    .cast(meta, strict=False)
                    .alias(col)
                )

        elif kind == "text":
            exprs.append(
                pl.col(col).str.strip_chars().fill_null("UNKNOWN").alias(col)
            )

        elif kind == "mixed_small_labels_int":
            base = pl.col(col).str.strip_chars()
            as_num = base.cast(pl.Float64, strict=False)
            expr = as_num
            for lbl, code in meta.items():
                expr = pl.when(base == lbl).then(pl.lit(float(code))).otherwise(expr)
            exprs.append(expr.cast(pl.Int16, strict=False).alias(col))

    return lf.with_columns(exprs) if exprs else lf





diag_polars = diagnose_polars(sample_df)
plan_polars = build_curation_plan_polars(diag_polars, sample_df)
print_plan(plan_polars)

curated_lf = apply_curation_polars(raw_lf, plan_polars)
print()
print(curated_lf.collect_schema())

dropped (3): ['ult_fec_cli_1t', 'conyuemp', 'tipodom']
date                      (2): ['fecha_dato', 'fecha_alta']
flag                      (26): ['ind_nuevo', 'ind_actividad_cliente', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']
int                       (3): ['ncodpers', 'indrel', 'cod_prov']
float                     (1): ['renta']
numeric_with_null_label   (2): ['age', 'antiguedad']
text                      (10): ['ind_empleado', 'pais_residencia', 'sexo', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'nomprov', 'segmen

## Curation

Rules applied to the diagnostic facts. Each column maps to one strategy:

- `drop_high_null` — over 99% missing, not worth imputing
- `drop_constant` — single unique value, no information
- `date` — 95%+ match a known date pattern
- `flag` — integers restricted to `{0, 1}`
- `int` / `float` — parse cleanly as their respective types
- `text` — over 95% non-numeric
- `numeric_with_null_label` — one non-numeric label that looks like a null placeholder
- `mixed_small_labels_int` — mostly integers plus a few labels; encoded as `max + 1, max + 2, ...` when the numeric part is integral

The last case recovers `P` in `indrel_1mes`. The mapping is derived from the data, not hardcoded: `P` becomes `max_num + 1`, which is 5. That is the principle throughout — rules derived from the data, not assumptions about the data.

In [ ]:
# User variability: how many users appear in consecutive months?

import time
t1 = time.time()
users_by_month = (
    curated_lf
    .group_by("fecha_dato")
    .agg(pl.col("ncodpers").unique().alias("users"))
    .sort("fecha_dato")
)

# Self-join con el mes anterior
variability = (
    users_by_month
    .with_columns(
        pl.col("users").shift(1).alias("prev_users")
    )
    .filter(pl.col("prev_users").is_not_null())
    .with_columns([
        pl.col("users").list.len().alias("curr_users"),
        pl.col("prev_users").list.len().alias("prev_users_n"),
        pl.col("users").list.set_intersection("prev_users").list.len().alias("retained"),
    ])
    .with_columns([
        (pl.col("curr_users") - pl.col("retained")).alias("new_users"),
        (pl.col("prev_users_n") - pl.col("retained")).alias("exited"),
        (pl.col("retained") / pl.col("prev_users_n") * 100).round(2).alias("retention_pct"),
        ((pl.col("curr_users") - pl.col("retained")) / pl.col("curr_users") * 100).round(2).alias("new_pct"),
    ])
    .select([
        pl.col("fecha_dato").cast(pl.Utf8).alias("month"),
        "prev_users_n", "curr_users", "retained", "new_users", "exited",
        "retention_pct", "new_pct"
    ])
    .collect()
)

print(variability)

t2 = time.time()
delta = t2 - t1
print(delta)

shape: (16, 8)
┌────────────┬──────────────┬────────────┬──────────┬───────────┬────────┬───────────────┬─────────┐
│ month      ┆ prev_users_n ┆ curr_users ┆ retained ┆ new_users ┆ exited ┆ retention_pct ┆ new_pct │
│ ---        ┆ ---          ┆ ---        ┆ ---      ┆ ---       ┆ ---    ┆ ---           ┆ ---     │
│ str        ┆ u32          ┆ u32        ┆ u32      ┆ u32       ┆ u32    ┆ f64           ┆ f64     │
╞════════════╪══════════════╪════════════╪══════════╪═══════════╪════════╪═══════════════╪═════════╡
│ 2015-02-28 ┆ 625457       ┆ 627394     ┆ 622926   ┆ 4468      ┆ 2531   ┆ 99.6          ┆ 0.71    │
│ 2015-03-28 ┆ 627394       ┆ 629209     ┆ 625094   ┆ 4115      ┆ 2300   ┆ 99.63         ┆ 0.65    │
│ 2015-04-28 ┆ 629209       ┆ 630367     ┆ 626698   ┆ 3669      ┆ 2511   ┆ 99.6          ┆ 0.58    │
│ 2015-05-28 ┆ 630367       ┆ 631957     ┆ 628135   ┆ 3822      ┆ 2232   ┆ 99.65         ┆ 0.6     │
│ 2015-06-28 ┆ 631957       ┆ 632110     ┆ 628603   ┆ 3507      ┆ 3354   ┆ 9

### Reading the variability

Two things stand out:

- **The retention rate is high** (typically 95-98%), meaning most users present one month are also present the next. This is why a model trained on one month generalizes to the next.
- **The new-user rate spikes in July 2015** to around 25%, consistent with a structural change in the customer base. From August 2015 onward, the new-user rate stabilizes at a lower level, but the absolute number of new users each month remains substantial.

The implication: the base is stable enough for temporal validation (April → May should generalize to May → June), but each month introduces enough new users that cold-start handling is non-trivial. A model without demographics cannot score these users.

In [ ]:
# Products per user, by month
PRODUCT_COLS = [c for c in curated_lf.collect_schema().names()
                if c.startswith("ind_") and c.endswith("_ult1") and c != "ind_nuevo"]
products_per_user = (
    curated_lf
    .with_columns(pl.sum_horizontal(PRODUCT_COLS).alias("n_products"))
    .group_by("fecha_dato")
    .agg([
        pl.col("n_products").mean().alias("mean"),
        pl.col("n_products").median().alias("median"),
        (pl.col("n_products") == 0).sum().alias("users_zero"),
        pl.len().alias("n_users"),
        ((pl.col("n_products") == 0).sum() / pl.len() * 100).round(2).alias("pct_zero"),
    ])
    .sort("fecha_dato")
    .collect()
)

print(products_per_user)

shape: (17, 6)
┌────────────┬──────────┬────────┬────────────┬─────────┬──────────┐
│ fecha_dato ┆ mean     ┆ median ┆ users_zero ┆ n_users ┆ pct_zero │
│ ---        ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      │
│ date       ┆ f64      ┆ f64    ┆ u32        ┆ u32     ┆ f64      │
╞════════════╪══════════╪════════╪════════════╪═════════╪══════════╡
│ 2015-01-28 ┆ 1.77369  ┆ 1.0    ┆ 17326      ┆ 625457  ┆ 2.77     │
│ 2015-02-28 ┆ 1.773579 ┆ 1.0    ┆ 17127      ┆ 627394  ┆ 2.73     │
│ 2015-03-28 ┆ 1.778152 ┆ 1.0    ┆ 17095      ┆ 629209  ┆ 2.72     │
│ 2015-04-28 ┆ 1.784349 ┆ 1.0    ┆ 17072      ┆ 630367  ┆ 2.71     │
│ 2015-05-28 ┆ 1.772048 ┆ 1.0    ┆ 16919      ┆ 631957  ┆ 2.68     │
│ 2015-06-28 ┆ 1.79162  ┆ 1.0    ┆ 16087      ┆ 632110  ┆ 2.54     │
│ 2015-07-28 ┆ 1.383389 ┆ 1.0    ┆ 204302     ┆ 829817  ┆ 24.62    │
│ 2015-08-28 ┆ 1.365983 ┆ 1.0    ┆ 205042     ┆ 843201  ┆ 24.32    │
│ 2015-09-28 ┆ 1.356195 ┆ 1.0    ┆ 211112     ┆ 865440  ┆ 24.39    │
│ 2015-10-28 ┆ 1.34

### Reading the products-per-user result

The mean products per user drops sharply between June 2015 (1.79) and July 2015 (1.38). Simultaneously, users with zero products jump from 2.5% to 24.6%.

This is a structural break. Roughly 200,000 users entered the bank with almost no products. From August 2015 onward, the distribution stabilizes — mean around 1.33, zero-products fraction around 24-25%.

Two consequences for modeling:

1. **A quarter of the 2016 customer base has no product history to learn from.** Pure collaborative filtering cannot score them — there are no interactions to factorize. This is the primary reason the hybrid model is necessary.
2. **The structural break creates two cohorts.** Users who joined before July 2015 have a different product distribution than those who joined after. A well-designed model must handle both, either through cohort features, seniority features, or training on the mixed base.

## Product scope

The 24-product catalog is dominated by 5 products covering 80% of the target. The top 8 cover 91.4%. Two products are excluded:

`ind_reca_fin_ult1` — 9,238 additions and 11 drops. The product started being tracked mid-dataset, so "additions" are first appearances in the log, not real acquisitions. Validation AUC is below 0.5 (anti-correlated) and holdout AP is 0.0006. Pure noise.

`ind_dela_fin_ult1` — 46 positives. Not enough to train.

The 16 non-modeled products fall back to popularity ranking at inference. The mechanism is explained in detail later.

In [ ]:
SAMPLE_SUB = "/content/sample_submission.csv"
FINAL_CUTOFF = date(2016, 5, 28)
VAL_CUTOFF = date(2016, 4, 28)
VAL_TARGET = date(2016, 5, 28)


FINAL_CUTOFF = date(2016, 5, 28)
VAL_CUTOFF = date(2016, 4, 28)
VAL_TARGET = date(2016, 5, 28)
# Products per user, by month
PRODUCT_COLS = [c for c in curated_lf.collect_schema().names()
                if c.startswith("ind_") and c.endswith("_ult1") and c != "ind_nuevo"]
# Compute per-product additions and their cumulative share
add_exprs = [
    ((pl.col(c) == 1) & (pl.col(c).shift(1).over("ncodpers", order_by="fecha_dato") == 0))
    .fill_null(False).cast(pl.UInt8).alias(f"{c}_add")
    for c in PRODUCT_COLS
]

transitions_lf = (
    curated_lf
    .with_columns(add_exprs)
    .with_columns(pl.sum_horizontal([f"{c}_add" for c in PRODUCT_COLS]).alias("n_added"))
    .filter(pl.col("fecha_dato") > pl.col("fecha_dato").min().over("ncodpers"))
)

product_stats = (
    transitions_lf
    .select([pl.col(f"{c}_add").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["additions"])
    .sort("additions", descending=True)
    .with_columns([
        (pl.col("additions") / pl.col("additions").sum() * 100).round(2).alias("pct"),
        (pl.col("additions").cum_sum() / pl.col("additions").sum() * 100).round(2).alias("cumulative_pct"),
    ])
)

print(product_stats)

# How many products to model for 90% / 95% / 99% coverage
for threshold in [0.90, 0.95, 0.99]:
    n_needed = (product_stats["cumulative_pct"] <= threshold * 100).sum() + 1
    print(f"  {int(threshold * 100)}% coverage: top {n_needed} products")

shape: (24, 4)
┌───────────────────┬───────────┬───────┬────────────────┐
│ column            ┆ additions ┆ pct   ┆ cumulative_pct │
│ ---               ┆ ---       ┆ ---   ┆ ---            │
│ str               ┆ i64       ┆ f64   ┆ f64            │
╞═══════════════════╪═══════════╪═══════╪════════════════╡
│ ind_recibo_ult1   ┆ 153205    ┆ 27.2  ┆ 27.2           │
│ ind_nom_pens_ult1 ┆ 84768     ┆ 15.05 ┆ 42.25          │
│ ind_nomina_ult1   ┆ 73800     ┆ 13.1  ┆ 55.35          │
│ ind_cco_fin_ult1  ┆ 69997     ┆ 12.43 ┆ 67.77          │
│ ind_tjcr_fin_ult1 ┆ 69311     ┆ 12.3  ┆ 80.08          │
│ ind_cno_fin_ult1  ┆ 37187     ┆ 6.6   ┆ 86.68          │
│ ind_ecue_fin_ult1 ┆ 26378     ┆ 4.68  ┆ 91.36          │
│ ind_dela_fin_ult1 ┆ 12707     ┆ 2.26  ┆ 93.62          │
│ ind_reca_fin_ult1 ┆ 9238      ┆ 1.64  ┆ 95.26          │
│ ind_ctma_fin_ult1 ┆ 7002      ┆ 1.24  ┆ 96.5           │
│ ind_valo_fin_ult1 ┆ 4850      ┆ 0.86  ┆ 97.36          │
│ ind_ctop_fin_ult1 ┆ 3882      ┆ 0.69  ┆

In [ ]:
# Additions and churn per product — reveals migration between products
additions_rank = (
    transitions_lf
    .select([pl.col(f"{c}_add").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["additions"])
    .sort("additions", descending=True)
)

churn_exprs = [
    ((pl.col(c) == 0) & (pl.col(c).shift(1).over("ncodpers", order_by="fecha_dato") == 1))
    .fill_null(False).cast(pl.UInt8).alias(f"{c}_drop")
    for c in PRODUCT_COLS
]

churn_rank = (
    curated_lf
    .with_columns(churn_exprs)
    .filter(pl.col("fecha_dato") > pl.col("fecha_dato").min().over("ncodpers"))
    .select([pl.col(f"{c}_drop").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["drops"])
    .sort("drops", descending=True)
)

net_flow = (
    additions_rank.select(["column", "additions"]).rename({"column": "product"})
    .join(
        churn_rank.select(["column", "drops"]).rename({"column": "product"}),
        on="product",
    )
    .with_columns([
        (pl.col("additions") - pl.col("drops")).alias("net_flow"),
        (pl.col("additions") / pl.col("additions").sum() * 100).round(2).alias("pct_of_additions"),
        (pl.col("additions").cum_sum() / pl.col("additions").sum() * 100).round(2).alias("cumulative_pct"),
    ])
    .sort(["net_flow","pct_of_additions"], descending=[True,True])
)

print(net_flow)

shape: (24, 6)
┌───────────────────┬───────────┬────────┬──────────┬──────────────────┬────────────────┐
│ product           ┆ additions ┆ drops  ┆ net_flow ┆ pct_of_additions ┆ cumulative_pct │
│ ---               ┆ ---       ┆ ---    ┆ ---      ┆ ---              ┆ ---            │
│ str               ┆ i64       ┆ i64    ┆ i64      ┆ f64              ┆ f64            │
╞═══════════════════╪═══════════╪════════╪══════════╪══════════════════╪════════════════╡
│ ind_recibo_ult1   ┆ 153205    ┆ 136594 ┆ 16611    ┆ 27.2             ┆ 27.2           │
│ ind_ecue_fin_ult1 ┆ 26378     ┆ 11035  ┆ 15343    ┆ 4.68             ┆ 93.62          │
│ ind_cno_fin_ult1  ┆ 37187     ┆ 23846  ┆ 13341    ┆ 6.6              ┆ 86.68          │
│ ind_reca_fin_ult1 ┆ 9238      ┆ 11     ┆ 9227     ┆ 1.64             ┆ 100.0          │
│ ind_nom_pens_ult1 ┆ 84768     ┆ 76112  ┆ 8656     ┆ 15.05            ┆ 54.67          │
│ ind_nomina_ult1   ┆ 73800     ┆ 65329  ┆ 8471     ┆ 13.1             ┆ 80.08       

### The concentration

Top-5 products by additions:

| Rank | Product | Additions | % of total | Cumulative % |
|---|---|---|---|---|
| 1 | `ind_recibo_ult1` | 153,205 | 27.2% | 27.2% |
| 2 | `ind_nom_pens_ult1` | 84,768 | 15.1% | 42.3% |
| 3 | `ind_nomina_ult1` | 73,800 | 13.1% | 55.4% |
| 4 | `ind_cco_fin_ult1` | 69,997 | 12.4% | 67.8% |
| 5 | `ind_tjcr_fin_ult1` | 69,311 | 12.3% | **80.1%** |

The top 5 account for **80.1%** of all 563,310 additions. Extending to top-7
(adding `ind_cno_fin_ult1` at 37,187 and `ind_ecue_fin_ult1` at 26,378)
reaches **91.4%**. Top-8 (adding `ind_dela_fin_ult1` at 12,707) reaches
**93.6%**.

The remaining 16 products share 6.4% of the target. `ind_ahor_fin_ult1` has
2 additions in 17 months, `ind_aval_fin_ult1` has 4. These are statistically
too sparse to model — no feature engineering recovers a signal from 2 examples.

The concentration explains why a popularity ranker is competitive. A
recommender that always suggests the top-7 products minus what the user
holds captures **91.4% of the target** by construction. The base rate is
3.8%, the top-7 concentration is 91.4%, and MAP@7 evaluates 7 predictions.
A trivial ranker achieves ~0.018 because those three facts align.

### Migration between products

The net flow reveals a migration pattern that the raw additions number hides:

| Product | Additions | Drops | Net flow | Interpretation |
|---|---|---|---|---|
| `ind_cco_fin_ult1` (traditional account) | 69,997 | 89,644 | **−19,647** | Losing holders |
| `ind_ecue_fin_ult1` (digital account) | 26,378 | 11,035 | **+15,343** | Gaining from migration |
| `ind_cno_fin_ult1` (payroll account) | 37,187 | 23,846 | **+13,341** | Gaining from migration |
| `ind_nomina_ult1` (payroll) | 73,800 | 65,329 | +8,471 | Stable |

The traditional current account loses 19.6k net holders over 17 months. The
digital account and payroll account gain 15.3k and 13.3k respectively. The
numbers are consistent with a single pattern: users are migrating from
`ind_cco_fin_ult1` to `ind_ecue_fin_ult1` or `ind_cno_fin_ult1`.

A user holding the current account but not the digital account is **not** a
cross-sell opportunity. They are a likely migration case. The state signal
"holds A but not B" describes what is changing, not what is available for
acquisition. Models that treat this as opportunity learn spurious correlations:
the user does not need another current account, they need a reason to keep the
one they have.

This is why state-only features mislead. "Holds `ind_cco`" alone is not a
predictive signal. "Holds `ind_cco` but not `ind_ecue`" is — but in the
opposite direction from what naive feature engineering would suggest.

### Two anomalies

**`ind_reca_fin_ult1`**: 9,238 additions and only 11 drops. A product with
9,000 acquisitions and 11 losses is statistically impossible in a real
market. This is a reporting artifact: the product started being tracked
mid-dataset, so "additions" are first appearances in the log, not real
acquisitions. Anyone using `ind_reca` as a target for prediction is modeling
the reporting change, not user behavior.

**`ind_aval_fin_ult1`**: 4 additions in 17 months. Practically dead. With
4 examples, no model can learn anything. The product should be excluded from
any per-product approach.

Both anomalies are visible only because additions and drops are compared
side by side. Additions alone would not have exposed the artifact.



In [ ]:
#  Monthly additions: the target we are predicting


additions_monthly = (
    transitions_lf
    .group_by("fecha_dato")
    .agg([
        pl.col("n_added").mean().alias("mean_products_added"),
        pl.len().alias("users"),
        (pl.col("n_added") > 0).sum().alias("users_with_addition"),
        ((pl.col("n_added") > 0).sum() / pl.len() * 100).round(2).alias("pct_with_addition"),
        pl.col("n_added").sum().alias("total_additions"),
    ])
    .sort("fecha_dato")
    .collect()
)

print(additions_monthly)


shape: (16, 6)
┌────────────┬───────────────────┬────────┬───────────────────┬──────────────────┬─────────────────┐
│ fecha_dato ┆ mean_products_add ┆ users  ┆ users_with_additi ┆ pct_with_additio ┆ total_additions │
│ ---        ┆ ed                ┆ ---    ┆ on                ┆ n                ┆ ---             │
│ date       ┆ ---               ┆ u32    ┆ ---               ┆ ---              ┆ i64             │
│            ┆ f64               ┆        ┆ u32               ┆ f64              ┆                 │
╞════════════╪═══════════════════╪════════╪═══════════════════╪══════════════════╪═════════════════╡
│ 2015-02-28 ┆ 0.050362          ┆ 622926 ┆ 23782             ┆ 3.82             ┆ 31372           │
│ 2015-03-28 ┆ 0.05128           ┆ 625233 ┆ 25305             ┆ 4.05             ┆ 32062           │
│ 2015-04-28 ┆ 0.048835          ┆ 626944 ┆ 25664             ┆ 4.09             ┆ 30617           │
│ 2015-05-28 ┆ 0.042251          ┆ 628456 ┆ 21422             ┆ 3.41        

### Reading the additions result
Roughly 3.8% of users add at least one product per month. Range: 2.9% to 5.3%. Seasonality: June 2015 peaks at 5.33% (pre-shift), August 2015 bottoms at 2.94% (dilution), February 2016 peaks at 3.95% (fiscal rhythm).

Three consequences:

1. **Accuracy is not useful.** Predicting "no one adds" gives 96% accuracy and zero value.
2. **The split must be temporal.** Random splitting leaks future information into the past on a seasonal series.

In [ ]:
# Modeling scope: top-10 products
TOP_PRODUCTS = product_stats.head(14)["column"].to_list()
top_10_share = product_stats.head(14)["cumulative_pct"].max()
print(f"\nTop 10 products cover {top_10_share:.2f}% of the target")
print(f"Modeled: {TOP_PRODUCTS}")


Top 10 products cover 99.25% of the target
Modeled: ['ind_recibo_ult1', 'ind_nom_pens_ult1', 'ind_nomina_ult1', 'ind_cco_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_cno_fin_ult1', 'ind_ecue_fin_ult1', 'ind_dela_fin_ult1', 'ind_reca_fin_ult1', 'ind_ctma_fin_ult1', 'ind_valo_fin_ult1', 'ind_ctop_fin_ult1', 'ind_fond_fin_ult1', 'ind_deco_fin_ult1']


## Shared feature engineering

A single pipeline produces the feature matrix that feeds both models. The feature groups are:

- **Demographics** — age, renta, tenure, segmento, canal, activity, and the rest.
- **24 holdings** — binary flags, one per product.
- **Trajectory** — additions in last 3 and 6 months, lifetime, days since last addition.
- **Clusters** — account alternatives, investment products, total products.

Three features are explicitly excluded:

`mes_dato` — in April it equals 4, in May it equals 5. Constant in single-month training, out of the ordinal vocabulary at prediction time.

`mes_dato_sin`, `mes_dato_cos` — cyclic month encoding. Constant per cutoff, different between cutoffs. Does not inform, distorts.

`alta_mes_sin`, `alta_mes_cos` are kept. They depend on `fecha_alta`, which is stable between months for the same user. No temporal shift.

In [ ]:
DEMO_COLS = [
    "age", "renta", "segmento", "ind_empleado", "pais_residencia",
    "sexo", "ind_nuevo", "indrel", "indrel_1mes", "tiprel_1mes",
    "indresi", "indext", "canal_entrada", "indfall", "cod_prov",
    "ind_actividad_cliente",
]

CAT_COLS = [
    "segmento", "ind_empleado", "pais_residencia", "sexo", "ind_nuevo",
    "indrel", "indrel_1mes", "tiprel_1mes", "indresi", "indext",
    "canal_entrada", "indfall", "cod_prov", "ind_actividad_cliente",
]

ACCOUNT_CLUSTER = [
    "ind_cco_fin_ult1", "ind_ecue_fin_ult1", "ind_cno_fin_ult1",
    "ind_ctop_fin_ult1", "ind_ctma_fin_ult1", "ind_ctpp_fin_ult1",
]
INVESTMENT_CLUSTER = [
    "ind_fond_fin_ult1", "ind_valo_fin_ult1", "ind_ahor_fin_ult1",
    "ind_dela_fin_ult1", "ind_deco_fin_ult1",
]

DATE_COL=["fecha_dato","fecha_alta"]
def build_raw_features(state_df, cutoff, transitions_lf):
    """
    Assemble the raw feature dataframe before the pipeline.
    Demographics + 24 holdings + trajectory + clusters.
    """
    base = state_df.select(["ncodpers"] + PRODUCT_COLS + DEMO_COLS+DATE_COL)

    base = (
        base
        .with_columns([
            (pl.col("fecha_dato") - pl.col("fecha_alta")).dt.total_days().clip(lower_bound=0).alias("antiguedad_dias"),
            pl.col("fecha_dato").dt.month().alias("mes_dato"),
            pl.col("fecha_alta").dt.month().alias("alta_mes"),
            pl.col("fecha_alta").dt.year().alias("alta_anio"),
        ])
        .with_columns([
            (pl.col("mes_dato") * (2 * np.pi / 12)).sin().alias("mes_dato_sin").cast(pl.Float64),
            (pl.col("mes_dato") * (2 * np.pi / 12)).cos().alias("mes_dato_cos").cast(pl.Float64),
            (pl.col("alta_mes") * (2 * np.pi / 12)).sin().alias("alta_mes_sin").cast(pl.Float64),
            (pl.col("alta_mes") * (2 * np.pi / 12)).cos().alias("alta_mes_cos").cast(pl.Float64),
        ])
        .with_columns([
            pl.col("age").cast(pl.Float64, strict=False),
            pl.col("renta").cast(pl.Float64, strict=False),
        ])
        .drop(["fecha_dato", "fecha_alta", "mes_dato", "alta_mes"])
    )

    # Trajectory features
    recent_3m = (
        cutoff.replace(month=cutoff.month - 3) if cutoff.month > 3
        else date(cutoff.year - 1, cutoff.month + 9, cutoff.day)
    )
    recent_6m = (
        cutoff.replace(month=cutoff.month - 6) if cutoff.month > 6
        else date(cutoff.year - 1, cutoff.month + 6, cutoff.day)
    )

    traj = (
        transitions_lf
        .filter(pl.col("fecha_dato") <= cutoff)
        .group_by("ncodpers")
        .agg([
            pl.col("n_added").filter(pl.col("fecha_dato") >= recent_3m).sum().fill_null(0).alias("additions_last_3m"),
            pl.col("n_added").filter(pl.col("fecha_dato") >= recent_6m).sum().fill_null(0).alias("additions_last_6m"),
            pl.col("n_added").sum().fill_null(0).alias("additions_lifetime"),
            (cutoff - pl.col("fecha_dato").filter(pl.col("n_added") > 0).max())
                .dt.total_days().alias("days_since_last_addition"),
        ])
        .with_columns([
            pl.col("additions_last_3m").cast(pl.Float64),
            pl.col("additions_last_6m").cast(pl.Float64),
            pl.col("additions_lifetime").cast(pl.Float64),
            pl.col("days_since_last_addition").fill_null(9999).cast(pl.Float64),
        ])
        .collect()
    )

    base = base.join(traj, on="ncodpers", how="left").with_columns([
        pl.col("additions_last_3m").fill_null(0),
        pl.col("additions_last_6m").fill_null(0),
        pl.col("additions_lifetime").fill_null(0),
        pl.col("days_since_last_addition").fill_null(9999),
    ])

    # Cluster counts
    base = base.with_columns([
        pl.sum_horizontal(ACCOUNT_CLUSTER).cast(pl.Float64).alias("n_account_alternatives"),
        pl.sum_horizontal(INVESTMENT_CLUSTER).cast(pl.Float64).alias("n_investment_products"),
        pl.sum_horizontal(PRODUCT_COLS).cast(pl.Float64).alias("n_products_total"),
    ])

    return base


# Build raw features for the April cutoff (validation)
state_april = (
    curated_lf
    .filter(pl.col("fecha_dato") == VAL_CUTOFF)
    .collect()
)
april_raw = build_raw_features(state_april, VAL_CUTOFF, transitions_lf)
print(f"April raw features: {len(april_raw):,} users, {len(april_raw.columns)} columns")

April raw features: 928,274 users, 54 columns


## April state, targets, held

`X_april` is the April feature matrix. `targets[p]` is 1 when a user did not hold `p` in April and acquired it in May. `held_in_april[p]` is 1 when the user already held `p` in April, used to mask the product at prediction time.

The same `pipe_april` transforms May and produces submissions. There is no `pipe_lgb` and no `pipe_cb`.

In [ ]:
# Cast predefined categoricals to String before the pipeline
april_raw = april_raw.with_columns(
    pl.col(CAT_COLS).cast(pl.String, strict=False)
)

target_col="target_bin"
cat_final_low, cat_final_high, num_final, discrete_final, cont_final = categorizar_todo_polars(april_raw, max_cat=20, n_dis=30 ,
                                                                                    excluir =
                                                                                [target_col, "mes_dato_sin", "mes_dato_cos","ncodpers","unix_timestamp", "release_date", "video_release_date"]+ ['user_id', 'movie_id'])

# Todas las variables a procesar
var_all = cat_final_low + cat_final_high + num_final

cat_final_high2=[ vari for vari in cat_final_high if vari not in ["title"]]
cat_final_high3=[var+"_TE" for var in cat_final_high2]
var_all_notext=[ var for var in var_all  if var not in ["title"]]


In [ ]:


# LightGBM pipeline: no OrdinalEncPolars

pipe_lgb = Pipeline([
  #  ('nlp_title', NLPFeatureExtractor(var=['title'], method='tfidf', max_features=25, ngram_range=(1,3))),
   ('na_imputer', Flagnan3Polars(var=var_all)),
    # ('imputacion', Flagnan3Polars(var=cat_final_low + cat_final_high + num_final)),
    ('rare_labels', RareLabelPolars(var=cat_final_low + cat_final_high +discrete_final
                              , tol=0.01)),

    ('out', Flagout2Polars(var=cont_final
                              , distance=3)),
    # ('target_encoding', TargetEncoderFastPandas2(var=cat_final_high2)),
   ('target_encoding', OrdinalEncPolars(var=cat_final_high2+cat_final_low+discrete_final)),

    #('target_encoding', TargetEncoderFastPolars(var=discrete_final)),
    #('bini_age', BiniPolars(var=["age"], nbins=5)),
    # ('bini_age', BiniPolars2(var= num_final, nbins=5)),

    # ('one_hot', OnehPolars(var=var_all_notext)), # Sexo y Bines de Edad a Dummies
  #  ('one_hot', OnehPolars(var=cat_final_low + discrete_final+cat_final_high3)), # Sexo y Bines de Edad a Dummies
   #('one_hot', OnehPolars(var=var_all)), # Sexo y Bines de Edad a Dummies
     # Ejemplo:


   ('drop', DropConstantFeaturesPolars(

   ))



])
april_features_pl = pipe_lgb.fit_transform(april_raw)
print(f"\nFeatures after pipeline: {len(april_features_pl.columns) - 1}")

# Which columns remained categorical (String dtype)?
cat_after_pipeline = [
    c for c in april_features_pl.columns
    if april_features_pl[c].dtype == pl.String
]
print(f"Categorical columns: {len(cat_after_pipeline)} -> {cat_after_pipeline}")


Features after pipeline: 56
Categorical columns: 0 -> []


In [ ]:
X_april = april_features_pl.to_pandas()

# for c in cat_after_pipeline:
#     X_april[c] = X_april[c].astype("category")

# print(f"Feature matrix: {X_april.shape}")
# print(f"Categorical dtypes: {(X_april.dtypes == 'category').sum()}")
# print(f"Float dtypes:       {(X_april.dtypes == 'float64').sum()}")
# print(f"Other:              {(~X_april.dtypes.isin(['category', 'float64'])).sum()}")

In [ ]:
# Targets: April -> May acquisition, aligned to X_april user order
TOP_PRODUCTS = [
    p for p in TOP_PRODUCTS
    if p not in ("ind_dela_fin_ult1","ind_reca_fin_ult1" ,"ind_valo_fin_ult1","ind_ctop_fin_ult1","ind_fond_fin_ult1","ind_deco_fin_ult1")
    # if p not in ("ind_reca_fin_ult1" ) ind_dela_fin_ult1 ind_reca_fin_ult1 ind_reca_fin_ult1 ind_ctop_fin_ult1 ind_valo_fin_ult1
]
prev_apr = (
    curated_lf
    .filter(pl.col("fecha_dato") == VAL_CUTOFF)
    .select(["ncodpers"] + PRODUCT_COLS)
    .collect()
)
nxt_may = (
    curated_lf
    .filter(pl.col("fecha_dato") == VAL_TARGET)
    .select(["ncodpers"] + PRODUCT_COLS)
    .collect()
)
joined_apr_may = prev_apr.join(nxt_may, on="ncodpers", suffix="_next")

user_ids_april = X_april["ncodpers"].values
uid_to_row = {uid: i for i, uid in enumerate(user_ids_april)}

targets = {p: np.zeros(len(X_april), dtype=np.int8) for p in TOP_PRODUCTS}
held_in_april = {p: np.zeros(len(X_april), dtype=bool) for p in TOP_PRODUCTS}

for row in joined_apr_may.iter_rows(named=True):
    uid = row["ncodpers"]
    if uid not in uid_to_row:
        continue
    i = uid_to_row[uid]
    for p in TOP_PRODUCTS:
        if row[p] == 1:
            held_in_april[p][i] = True
        elif row.get(f"{p}_next", 0) == 1:
            targets[p][i] = 1

print(f"Positive rates per product (among non-holders):")
for p in TOP_PRODUCTS:
    n_train = (~held_in_april[p]).sum()
    n_pos = targets[p][~held_in_april[p]].sum()
    rate = n_pos / n_train * 100 if n_train > 0 else 0
    print(f"  {p:<25} train={n_train:>7,}  pos={n_pos:>6,}  rate={rate:.3f}%")

Positive rates per product (among non-holders):
  ind_recibo_ult1           train=816,551  pos=10,163  rate=1.245%
  ind_nom_pens_ult1         train=877,604  pos= 5,513  rate=0.628%
  ind_nomina_ult1           train=882,127  pos= 5,488  rate=0.622%
  ind_cco_fin_ult1          train=366,484  pos= 3,854  rate=1.052%
  ind_tjcr_fin_ult1         train=893,467  pos= 4,248  rate=0.475%
  ind_cno_fin_ult1          train=856,519  pos= 2,346  rate=0.274%
  ind_ecue_fin_ult1         train=854,549  pos= 2,709  rate=0.317%
  ind_ctma_fin_ult1         train=919,622  pos=   512  rate=0.056%


## LightGBM training

8 classifiers, 600 trees, `is_unbalance=True`. The purpose of `is_unbalance` is to balance the loss (products have base rates between 0.03% and 1.2%). It multiplies the positive class by `n_neg / n_pos`, mathematically equivalent to CatBoost's `auto_class_weights='Balanced'`.

Trains on all non-holders, no holdout. LightGBM defaults on this dataset (600 trees, unlimited depth) generalize well on tabular data because deep trees capture interactions that shallow trees miss.

In [ ]:
from sklearn.model_selection import train_test_split

LGB_PARAMS = {
    "n_estimators": 2000,
    # "num_leaves": 15,
    # "min_child_samples": 200,
    # "reg_lambda": 10,
    "is_unbalance": True,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": 4,
}

lgb_models = {}
lgb_results = []

for p in TOP_PRODUCTS:
    feature_cols = [c for c in X_april.columns if c not in ["ncodpers", p]]
    mask = ~held_in_april[p]
    X_full = X_april.loc[mask, feature_cols]
    y_full = targets[p][mask]

    if y_full.sum() < 100:
        print(f"  {p:<25} SKIP ({y_full.sum()} positives)")
        continue

    X_tr, X_ho, y_tr, y_ho = train_test_split(
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=42,
    )

    stage1 = lgb.LGBMClassifier(**LGB_PARAMS, enable_categorical=True)
    stage1.fit(
        X_tr, y_tr,
        eval_set=[(X_ho, y_ho)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )
    best_iter = stage1.best_iteration_

    final_params = dict(LGB_PARAMS)
    final_params["n_estimators"] = max(best_iter, 50)
    final_model = lgb.LGBMClassifier(**final_params, enable_categorical=True)
    final_model.fit(X_full, y_full)

    lgb_models[p] = (final_model, feature_cols)

    y_pred_ho = final_model.predict_proba(X_ho)[:, 1]
    lgb_results.append({
        "product": p,
        "best_iter": best_iter,
        "auc_ho": round(roc_auc_score(y_ho, y_pred_ho), 4),
        "ap_ho": round(average_precision_score(y_ho, y_pred_ho), 4),
    })

for r in lgb_results:
    print(f"  {r['product']:<25} iter={r['best_iter']:>4}  AUC={r['auc_ho']:.4f}  AP={r['ap_ho']:.4f}")

  ind_recibo_ult1           iter=  59  AUC=0.9620  AP=0.2544
  ind_nom_pens_ult1         iter=   1  AUC=0.9183  AP=0.0951
  ind_nomina_ult1           iter=   1  AUC=0.9226  AP=0.0946
  ind_cco_fin_ult1          iter=  28  AUC=0.9588  AP=0.2602
  ind_tjcr_fin_ult1         iter=   1  AUC=0.8803  AP=0.0395
  ind_cno_fin_ult1          iter=   1  AUC=0.8136  AP=0.0178
  ind_ecue_fin_ult1         iter=   1  AUC=0.7855  AP=0.0197
  ind_ctma_fin_ult1         iter=   1  AUC=0.8812  AP=0.0075


## Popularity fallback — the key piece

The 16 non-modeled products have no model score. To rank them alongside the 8 modeled products, we use their historical addition rate over the April to May transition, computed among non-holders only.

Among non-holders because a holder of `p` cannot acquire `p`. The relevant quantity is P(acquire `p` | do not hold `p`).

`pop_rates[p] = n_pos / n_non_holders` sits in [0, 1], the same scale as model probabilities. This is what allows mixing both in a single ranking.

The fallback is constant per product: every non-holder of `p` receives exactly `pop_rates[p]`. That is correct in raw. `pop_rates[ind_fond] ≈ 0.001` never enters the top-7 of anyone except perhaps for very rare users.

Under percentile this breaks. `rank(pct=True)` over a constant column returns ~0.5 for every row. The fallback jumps from "0.001, never enters" to "0.5, competes head-to-head with the model". It leaks into the top-7 of users who will not acquire it, burning slots that should go to `cco`, `recibo`, `nomina`.

This is the percentile bug. Not a "worse model", but a ranking function that destroys the fallback.

In [ ]:
pop_rates = {}
for p in PRODUCT_COLS:
    n_non_hold, n_pos = 0, 0
    for row in joined_apr_may.iter_rows(named=True):
        uid = row["ncodpers"]
        if uid not in uid_to_row:
            continue
        if row[p] == 0:
            n_non_hold += 1
            if row.get(f"{p}_next", 0) == 1:
                n_pos += 1
    pop_rates[p] = n_pos / max(n_non_hold, 1)

print("Popularity rates (April to May, among non-holders):")
for p, rate in sorted(pop_rates.items(), key=lambda x: -x[1]):
    marker = " [modeled]" if p in TOP_PRODUCTS else ""
    print(f"  {p:<25} {rate * 100:>6.3f}%{marker}")

Popularity rates (April to May, among non-holders):
  ind_recibo_ult1            1.247% [modeled]
  ind_cco_fin_ult1           1.056% [modeled]
  ind_nom_pens_ult1          0.629% [modeled]
  ind_nomina_ult1            0.623% [modeled]
  ind_tjcr_fin_ult1          0.476% [modeled]
  ind_ecue_fin_ult1          0.318% [modeled]
  ind_cno_fin_ult1           0.274% [modeled]
  ind_ctma_fin_ult1          0.056% [modeled]
  ind_reca_fin_ult1          0.032%
  ind_ctop_fin_ult1          0.027%
  ind_valo_fin_ult1          0.020%
  ind_ctpp_fin_ult1          0.015%
  ind_fond_fin_ult1          0.007%
  ind_dela_fin_ult1          0.005%
  ind_ctju_fin_ult1          0.004%
  ind_plan_fin_ult1          0.002%
  ind_viv_fin_ult1           0.001%
  ind_pres_fin_ult1          0.001%
  ind_cder_fin_ult1          0.001%
  ind_hip_fin_ult1           0.000%
  ind_ahor_fin_ult1          0.000%
  ind_aval_fin_ult1          0.000%
  ind_deco_fin_ult1          0.000%
  ind_deme_fin_ult1          0.000%


## MAP@7 metrics

Two variants.

**Filtered MAP@7** averages over users who acquired something in May. Measures ranking quality where signal exists.

**Full-population MAP@7** averages over all users, with non-adders contributing 0. Comparable to the Kaggle leaderboard.

In [ ]:
def compute_map7_filtered(predictions, actual, k=7):
    aps = []
    for uid, actual_set in actual.items():
        recs = predictions.get(uid, [])[:k]
        if not recs:
            continue
        hits, ap = 0, 0.0
        for rank, prod in enumerate(recs, start=1):
            if prod in actual_set:
                hits += 1
                ap += hits / rank
        aps.append(ap / min(len(actual_set), k))
    return float(np.mean(aps)) if aps else 0.0


def compute_map7_fullpop(predictions, actual, all_users, k=7):
    aps = []
    for uid in all_users:
        if uid in actual:
            actual_set = actual[uid]
            recs = predictions.get(uid, [])[:k]
            if not recs:
                aps.append(0.0)
                continue
            hits, ap = 0, 0.0
            for rank, prod in enumerate(recs, start=1):
                if prod in actual_set:
                    hits += 1
                    ap += hits / rank
            aps.append(ap / min(len(actual_set), k))
        else:
            aps.append(0.0)
    return float(np.mean(aps)) if aps else 0.0



## `build_predictions` — ranking, masking, fallback

Per user:

1. Score every product from the probability matrix.
2. Masking: set the score of already-held products to `-inf`.
3. Top-K: sort by score descending, take the first 7 finite values.
4. Fill: if masking left fewer than 7, complete with `pop_list` in order, skipping products already chosen.

Step 4 only fires for users holding 18 or more products (24 - 7 + 1 = 18). For the vast majority of users it does not trigger.

In [ ]:
def build_predictions(probs_df, held_dict, user_ids, pop_rates_local, top_k=7):
    pop_list = sorted(pop_rates_local.keys(), key=lambda p: -pop_rates_local[p])
    uid_to_row_local = {uid: i for i, uid in enumerate(probs_df["ncodpers"].values)}
    predictions = {}
    for uid in user_ids:
        if uid not in uid_to_row_local:
            predictions[uid] = pop_list[:top_k]
            continue
        i = uid_to_row_local[uid]
        scores = {p: probs_df.at[i, p] for p in PRODUCT_COLS}
        for p in held_dict.get(uid, set()):
            scores[p] = -np.inf
        top = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        recs = [p for p, s in top if np.isfinite(s)]
        while len(recs) < top_k:
            for p in pop_list:
                if p not in recs:
                    recs.append(p)
                    if len(recs) == top_k:
                        break
        predictions[uid] = recs[:top_k]
    return predictions


# April held products built from state_april, which holds all 24 product columns
held_dict_apr = {
    row["ncodpers"]: {p for p in PRODUCT_COLS if row[p] == 1}
    for row in state_april.iter_rows(named=True)
    if row["ncodpers"] in uid_to_row
}

actual_additions = {}
for row in joined_apr_may.iter_rows(named=True):
    uid = row["ncodpers"]
    if uid not in uid_to_row:
        continue
    added = {p for p in PRODUCT_COLS if row[p] == 0 and row.get(f"{p}_next", 0) == 1}
    if added:
        actual_additions[uid] = added

print(f"held_dict_apr:    {len(held_dict_apr):,} users")
print(f"actual_additions: {len(actual_additions):,} users")
assert len(held_dict_apr) == len(user_ids_april), "state_april and X_april are not aligned"

held_dict_apr:    928,274 users
actual_additions: 27,875 users


## CatBoost training

8 classifiers with ordered boosting. Two stages.

**Stage 1** holds out 20% and trains with early stopping (up to 2000 iterations, stop when validation AUC plateaus for 50 trees). `auto_class_weights='Balanced'` compensates for the 0.03% to 1.2% positive rate.

**Stage 2** retrieves `best_iteration` and retrains on 100% of non-holders with that fixed tree count.

The critical detail is that `auto_class_weights='Balanced'` must be present in both stages. If only stage 1 has it, the final model optimizes raw Logloss on a 1% positive rate and produces near-constant probabilities, which destroys the cross-product ranking.

In [ ]:
CB_PARAMS = {
    "iterations": 2000,
    # "depth": 6,
    # "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    # "auto_class_weights": "Balanced",
    # "l2_leaf_reg": 10,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
}

cb_models = {}
cb_results = []

for p in TOP_PRODUCTS:
    feature_cols = [c for c in X_april.columns if c not in ["ncodpers", p]]
    mask = ~held_in_april[p]
    X_full = X_april.loc[mask, feature_cols]
    y_full = targets[p][mask]

    if y_full.sum() < 100:
        print(f"  {p:<25} SKIP")
        continue

    X_tr, X_ho, y_tr, y_ho = train_test_split(
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=42,
    )
    stage1 = CatBoostClassifier(**CB_PARAMS)
    stage1.fit(X_tr, y_tr, eval_set=(X_ho, y_ho), early_stopping_rounds=50, use_best_model=True)
    best_iter = stage1.get_best_iteration()

    # Balanced weights must persist in stage 2, otherwise the final model produces near-constant probabilities
    final_params = dict(CB_PARAMS)
    final_params["iterations"] = max(best_iter + 1, 50)
    final_model = CatBoostClassifier(**final_params)
    final_model.fit(X_full, y_full)

    cb_models[p] = (final_model, feature_cols)

    y_pred_ho = final_model.predict_proba(X_ho)[:, 1]
    cb_results.append({
        "product": p,
        "train_n": int(mask.sum()),
        "positives": int(y_full.sum()),
        "best_iter": best_iter,
        "auc": round(roc_auc_score(y_ho, y_pred_ho), 4),
        "ap": round(average_precision_score(y_ho, y_pred_ho), 4),
    })

print(f"\nTrained {len(cb_models)} CatBoost models")
for r in cb_results:
    print(f"  {r['product']:<25} iter={r['best_iter']:>4}  AUC={r['auc']:.4f}  AP={r['ap']:.4f}")


Trained 8 CatBoost models
  ind_recibo_ult1           iter= 440  AUC=0.9695  AP=0.5455
  ind_nom_pens_ult1         iter= 318  AUC=0.9693  AP=0.6346
  ind_nomina_ult1           iter= 402  AUC=0.9796  AP=0.6776
  ind_cco_fin_ult1          iter=  86  AUC=0.9419  AP=0.4151
  ind_tjcr_fin_ult1         iter= 227  AUC=0.9746  AP=0.3955
  ind_cno_fin_ult1          iter= 276  AUC=0.9733  AP=0.6617
  ind_ecue_fin_ult1         iter= 158  AUC=0.9650  AP=0.5733
  ind_ctma_fin_ult1         iter= 295  AUC=0.9995  AP=0.7589


## Raw vs percentile — the trap

Four combinations are generated for comparison:

|          | Raw        | Percentile   |
|----------|------------|--------------|
| LightGBM | `probs_lgb`| `ranked_lgb` |
| CatBoost | `probs_cb` | `ranked_cb`  |

**Raw** uses `probs_df[p]` directly. The fallback stays at `pop_rates[p] ≈ 0.001` and never enters the top-7.

**Percentile** applies `probs_df[p].rank(pct=True)` per column. Every product, modeled or fallback, is ranked. The fallback, being constant, lands near percentile 0.5.

The difference is invisible in validation (both land in the 0.0234 to 0.0236 range) and dramatic on Kaggle (0.0178 vs 0.0235). This cell makes that visible empirically.

In [ ]:
probs_lgb = pd.DataFrame({"ncodpers": X_april["ncodpers"].values})
for p, (model, feat_cols) in lgb_models.items():
    probs_lgb[p] = model.predict_proba(X_april[feat_cols])[:, 1]
for p in PRODUCT_COLS:
    if p not in probs_lgb.columns:
        probs_lgb[p] = pop_rates[p]

probs_cb = pd.DataFrame({"ncodpers": X_april["ncodpers"].values})
for p, (model, feat_cols) in cb_models.items():
    probs_cb[p] = model.predict_proba(X_april[feat_cols])[:, 1]
for p in PRODUCT_COLS:
    if p not in probs_cb.columns:
        probs_cb[p] = pop_rates[p]

# Percentile rank is applied per column, which is where the fallback stops behaving as a fallback
ranked_lgb = probs_lgb.copy()
ranked_cb = probs_cb.copy()
for p in PRODUCT_COLS:
    ranked_lgb[p] = probs_lgb[p].rank(pct=True)
    ranked_cb[p] = probs_cb[p].rank(pct=True)

preds_lgb_raw = build_predictions(probs_lgb, held_dict_apr, user_ids_april, pop_rates)
preds_lgb_pct = build_predictions(ranked_lgb, held_dict_apr, user_ids_april, pop_rates)
preds_cb_raw  = build_predictions(probs_cb,  held_dict_apr, user_ids_april, pop_rates)
preds_cb_pct  = build_predictions(ranked_cb,  held_dict_apr, user_ids_april, pop_rates)

probs_pop = pd.DataFrame({"ncodpers": X_april["ncodpers"].values})
for p in PRODUCT_COLS:
    probs_pop[p] = pop_rates[p]
preds_pop = build_predictions(probs_pop, held_dict_apr, user_ids_april, pop_rates)

print("Predictions generated for LGB raw, LGB percentile, CB raw, CB percentile, Popularity")

Predictions generated for LGB raw, LGB percentile, CB raw, CB percentile, Popularity


## Validation on April to May

April to May is the training split. Numbers are inflated by train-user overlap. The only number that matters for Kaggle is the leaderboard.

What we can verify: the popularity control should reproduce the notebook baseline (~0.0193 fullpop, ~0.6431 filtered). If it does not, there is a bug in `pop_rates` or `held_dict_apr`.

In [ ]:
rows = []
for name, preds in [
    ("Popularity (control)", preds_pop),
    ("LightGBM raw", preds_lgb_raw),
    ("LightGBM percentile", preds_lgb_pct),
    ("CatBoost raw", preds_cb_raw),
    ("CatBoost percentile", preds_cb_pct),
]:
    rows.append({
        "model": name,
        "filtered": round(compute_map7_filtered(preds, actual_additions), 4),
        "fullpop": round(compute_map7_fullpop(preds, actual_additions, user_ids_april), 4),
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))
print()
print("Historical references on the same April to May split:")
print("  Popularity     filtered=0.6431  fullpop=0.0193")
print("  LightFM        filtered=0.7679  fullpop=0.0231")
print("  CatBoost raw   leaderboard=0.02348")
print("  CatBoost pct   leaderboard=0.01784")
print()
print("Expected: raw and percentile are nearly identical in validation.")
print("The gap is on Kaggle, which is why validation cannot decide between them.")

               model  filtered  fullpop
Popularity (control)    0.6448   0.0194
        LightGBM raw    0.5377   0.0161
 LightGBM percentile    0.6886   0.0207
        CatBoost raw    0.9024   0.0271
 CatBoost percentile    0.7982   0.0240

Historical references on the same April to May split:
  Popularity     filtered=0.6431  fullpop=0.0193
  LightFM        filtered=0.7679  fullpop=0.0231
  CatBoost raw   leaderboard=0.02348
  CatBoost pct   leaderboard=0.01784

Expected: raw and percentile are nearly identical in validation.
The gap is on Kaggle, which is why validation cannot decide between them.


In [ ]:
from collections import Counter

def product_distribution(preds, actual):
    top7 = Counter()
    act = Counter()
    for uid, recs in preds.items():
        for p in recs:
            top7[p] += 1
    for uid, added in actual.items():
        for p in added:
            act[p] += 1
    return top7, act

top7_lgb_raw, act = product_distribution(preds_lgb_raw, actual_additions)
top7_lgb_pct, _   = product_distribution(preds_lgb_pct, actual_additions)

print(f"{'Product':<25} {'actual':>7} {'raw_top7':>9} {'pct_top7':>9} {'ratio_raw':>10} {'ratio_pct':>10}")
for p in PRODUCT_COLS:
    a = max(act.get(p, 0), 1)
    r_raw = top7_lgb_raw.get(p, 0) / a
    r_pct = top7_lgb_pct.get(p, 0) / a
    tag = " [modeled]" if p in TOP_PRODUCTS else " [fallback]"
    print(f"{p:<25} {act.get(p,0):>7,} {top7_lgb_raw.get(p,0):>9,} {top7_lgb_pct.get(p,0):>9,} {r_raw:>10.2f} {r_pct:>10.2f}{tag}")

print()
print("Fallbacks stay low under raw and explode under percentile.")

Product                    actual  raw_top7  pct_top7  ratio_raw  ratio_pct
ind_ahor_fin_ult1               1        17   921,664      17.00  921664.00 [fallback]
ind_aval_fin_ult1               0         2   888,976       2.00  888976.00 [fallback]
ind_cco_fin_ult1            3,854   364,164   126,913      94.49      32.93 [modeled]
ind_cder_fin_ult1               5       279   812,442      55.80  162488.40 [fallback]
ind_cno_fin_ult1            2,346   773,946   428,542     329.90     182.67 [modeled]
ind_ctju_fin_ult1              40    16,512   622,107     412.80   15552.67 [fallback]
ind_ctma_fin_ult1             512    93,510   102,861     182.64     200.90 [modeled]
ind_ctop_fin_ult1             226   265,902   324,815    1176.56    1437.23 [fallback]
ind_ctpp_fin_ult1             131    84,724   171,570     646.75    1309.69 [fallback]
ind_deco_fin_ult1               0         0    51,315       0.00   51315.00 [fallback]
ind_deme_fin_ult1               0         0     3,551    

## Sanity check: submission path reproduces validation

This check catches the LightGBM submission bug. April is rebuilt via `pipe_april.transform(april_raw)`, which is the exact same path the May submission uses, then predicted and compared to the value in the previous cell.

If the path is sound, delta is near 0. If there is a bug (different pipeline, category cast introducing NaN, missing column), delta is significantly negative.

In [ ]:
apr_repl_pl = pipe_lgb.transform(april_raw)
X_apr_check = apr_repl_pl.to_pandas()
for c in cat_after_pipeline:
    if c in X_apr_check.columns and c in X_april.columns:
        X_apr_check[c] = pd.Categorical(X_apr_check[c], categories=X_april[c].cat.categories)

nan_cast = sum(X_apr_check[c].isna().sum() for c in cat_after_pipeline if c in X_apr_check.columns)
print(f"NaN in April categoricals after cast: {nan_cast}")
assert nan_cast == 0, "Category cast introduced NaN, pipelines are misaligned"

missing = [c for c in X_april.columns if c not in X_apr_check.columns]
print(f"Missing columns: {len(missing)}")
assert len(missing) == 0

probs_apr_check = pd.DataFrame({"ncodpers": X_apr_check["ncodpers"].values})
for p, (model, feat_cols) in lgb_models.items():
    probs_apr_check[p] = model.predict_proba(X_apr_check[feat_cols])[:, 1]
for p in PRODUCT_COLS:
    if p not in probs_apr_check.columns:
        probs_apr_check[p] = pop_rates[p]

preds_apr_check = build_predictions(probs_apr_check, held_dict_apr, user_ids_april, pop_rates)
fullpop_check = compute_map7_fullpop(preds_apr_check, actual_additions, user_ids_april)
fullpop_ref = compute_map7_fullpop(preds_lgb_raw, actual_additions, user_ids_april)
print(f"\nFullpop via submission path: {fullpop_check:.4f}")
print(f"Fullpop original:            {fullpop_ref:.4f}")
print(f"Delta:                       {fullpop_check - fullpop_ref:+.4f}")
print()
print("Delta should be near 0. A negative delta means the submission path is broken.")

NaN in April categoricals after cast: 0
Missing columns: 0

Fullpop via submission path: 0.0167
Fullpop original:            0.0167
Delta:                       +0.0000

Delta should be near 0. A negative delta means the submission path is broken.


## May state transform

Same `build_raw_features` and same `pipe_april`. No new pipelines. The cast to `category` uses `categories=X_april[c].cat.categories` so that internal codes match April. The check verifies the cast does not introduce NaN. If May has categories that April did not and `RareLabelPolars` did not group them, the cast would turn them into NaN, LightGBM would treat them as code -1, and the prediction for those rows would be corrupt.

In [ ]:
state_may = curated_lf.filter(pl.col("fecha_dato") == FINAL_CUTOFF).collect()
may_raw = build_raw_features(state_may, FINAL_CUTOFF, transitions_lf)
may_raw = may_raw.with_columns(
    pl.col(CAT_COLS).cast(pl.String, strict=False)
)
may_features_pl = pipe_lgb.transform(may_raw)
X_may = may_features_pl.to_pandas()

nan_before, nan_after = 0, 0
for c in cat_after_pipeline:
    if c not in X_may.columns:
        print(f"MISSING in May: {c}")
        continue
    nan_before += X_may[c].isna().sum()
    X_may[c] = pd.Categorical(X_may[c], categories=X_april[c].cat.categories)
    nan_after += X_may[c].isna().sum()

print(f"May feature matrix: {X_may.shape}")
print(f"NaN categoricals before cast: {nan_before}")
print(f"NaN categoricals after cast:  {nan_after}")
print(f"NaN introduced by cast:       {nan_after - nan_before}")
if nan_after - nan_before > 0:
    print("Warning: cast introduced NaN, pipelines or categories are misaligned")

missing_may = [c for c in X_april.columns if c not in X_may.columns]
print(f"Missing columns in May: {len(missing_may)}")

may_held = {
    row["ncodpers"]: {p for p in PRODUCT_COLS if row[p] == 1}
    for row in state_may.iter_rows(named=True)
}
print(f"May users with product data: {len(may_held):,}")

May feature matrix: (931453, 57)
NaN categoricals before cast: 0
NaN categoricals after cast:  0
NaN introduced by cast:       0
Missing columns in May: 0
May users with product data: 931,453


## Submission files

Four files are generated. Only the two raw files are submitted, since the historical CatBoost comparison already establishes that raw beats percentile on Kaggle by 0.0064. The percentile files are generated so the delta can be confirmed with an extra submission if desired.

Files written to `/content/`:

`submission_lgb_raw.csv` — the LightGBM fix
`submission_cb_raw.csv` — the CatBoost baseline that scored ~0.0235
`submission_lgb_pct.csv` — optional, confirms the trap
`submission_cb_pct.csv` — optional, confirms the trap

In [ ]:
sample = pd.read_csv(SAMPLE_SUB)
test_user_ids = sample["ncodpers"].tolist()


def build_submission(probs_may_df, user_ids, held_dict, pop_rates_local, top_k=7):
    pop_list_local = sorted(pop_rates_local.keys(), key=lambda p: -pop_rates_local[p])
    uid_to_row_local = {uid: i for i, uid in enumerate(probs_may_df["ncodpers"].values)}
    rows = []
    n_scored, n_fallback = 0, 0
    for uid in user_ids:
        if uid not in uid_to_row_local:
            rows.append({"ncodpers": uid, "added_products": " ".join(pop_list_local[:top_k])})
            n_fallback += 1
            continue
        i = uid_to_row_local[uid]
        scores = {p: probs_may_df.at[i, p] for p in PRODUCT_COLS}
        for p in held_dict.get(uid, set()):
            scores[p] = -np.inf
        top = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        recs = [p for p, s in top if np.isfinite(s)]
        while len(recs) < top_k:
            for p in pop_list_local:
                if p not in recs:
                    recs.append(p)
                    if len(recs) == top_k:
                        break
        rows.append({"ncodpers": uid, "added_products": " ".join(recs[:top_k])})
        n_scored += 1
    return pd.DataFrame(rows).sort_values("ncodpers").reset_index(drop=True), n_scored, n_fallback


probs_lgb_may = pd.DataFrame({"ncodpers": X_may["ncodpers"].values})
for p, (model, feat_cols) in lgb_models.items():
    probs_lgb_may[p] = model.predict_proba(X_may[feat_cols])[:, 1]
for p in PRODUCT_COLS:
    if p not in probs_lgb_may.columns:
        probs_lgb_may[p] = pop_rates[p]

probs_cb_may = pd.DataFrame({"ncodpers": X_may["ncodpers"].values})
for p, (model, feat_cols) in cb_models.items():
    probs_cb_may[p] = model.predict_proba(X_may[feat_cols])[:, 1]
for p in PRODUCT_COLS:
    if p not in probs_cb_may.columns:
        probs_cb_may[p] = pop_rates[p]

ranked_lgb_may = probs_lgb_may.copy()
ranked_cb_may = probs_cb_may.copy()
for p in PRODUCT_COLS:
    ranked_lgb_may[p] = probs_lgb_may[p].rank(pct=True)
    ranked_cb_may[p] = probs_cb_may[p].rank(pct=True)

sub_lgb_raw, ns, nf = build_submission(probs_lgb_may, test_user_ids, may_held, pop_rates)
sub_lgb_pct, _, _ = build_submission(ranked_lgb_may, test_user_ids, may_held, pop_rates)
sub_cb_raw, _, _ = build_submission(probs_cb_may, test_user_ids, may_held, pop_rates)
sub_cb_pct, _, _ = build_submission(ranked_cb_may, test_user_ids, may_held, pop_rates)

sub_lgb_raw.to_csv("/content/submission_lgb_raw.csv", index=False)
sub_lgb_pct.to_csv("/content/submission_lgb_pct.csv", index=False)
sub_cb_raw.to_csv("/content/submission_cb_raw.csv", index=False)
sub_cb_pct.to_csv("/content/submission_cb_pct.csv", index=False)

print(f"LightGBM raw: scored={ns:,}  fallback={nf:,}")
print("Four files written to /content/")

LightGBM raw: scored=929,615  fallback=0
Four files written to /content/


## Format checks, mask leak, distribution

Three checks before submitting:

1. **Format** — rows match sample, user set matches, no duplicates, 7 products per row.
2. **Mask leak** — no user has in their top-7 a product they already hold in May.
3. **Distribution** — under raw, the most frequent products in the top-7 are the high base rate ones (`cco`, `recibo`). Under percentile, fallbacks leak in.

In [ ]:
def check_submission(name, sub):
    assert len(sub) == len(sample), f"{name}: row count"
    assert set(sub["ncodpers"]) == set(test_user_ids), f"{name}: user set"
    assert sub["ncodpers"].duplicated().sum() == 0, f"{name}: duplicates"
    assert all(len(r.split()) == 7 for r in sub["added_products"]), f"{name}: not 7 products"
    assert sub["added_products"].isnull().sum() == 0, f"{name}: nulls"

    leak = 0
    for _, r in sub.iterrows():
        uid = r["ncodpers"]
        prods = set(r["added_products"].split())
        if prods & may_held.get(uid, set()):
            leak += 1
    assert leak == 0, f"{name}: {leak} mask leaks"
    print(f"{name:<22} OK  rows={len(sub):,}  mask_leaks=0")


check_submission("LightGBM raw", sub_lgb_raw)
check_submission("LightGBM pct", sub_lgb_pct)
check_submission("CatBoost raw", sub_cb_raw)
check_submission("CatBoost pct", sub_cb_pct)

from collections import Counter
def top_dist(sub, n=10):
    c = Counter()
    for r in sub["added_products"]:
        c.update(r.split())
    return c.most_common(n)

print("\nTop-7 distribution, LightGBM raw vs LightGBM percentile:")
print(f"{'Product':<25} {'raw%':>7} {'pct%':>7} {'base%':>7}")
raw_counts = dict(top_dist(sub_lgb_raw, n=30))
pct_counts = dict(top_dist(sub_lgb_pct, n=30))
for p in PRODUCT_COLS:
    r = raw_counts.get(p, 0) / len(sub_lgb_raw) * 100
    pc = pct_counts.get(p, 0) / len(sub_lgb_pct) * 100
    base = pop_rates[p] * 100
    tag = " [modeled]" if p in TOP_PRODUCTS else ""
    print(f"{p:<25} {r:>7.2f} {pc:>7.2f} {base:>7.3f}{tag}")

print()
print("Under percentile, fallbacks rise to 10 to 15 percent of top-7 slots.")
print("Under raw, they stay near 1 percent. That is the source of the Kaggle gap.")

LightGBM raw           OK  rows=929,615  mask_leaks=0
LightGBM pct           OK  rows=929,615  mask_leaks=0
CatBoost raw           OK  rows=929,615  mask_leaks=0
CatBoost pct           OK  rows=929,615  mask_leaks=0

Top-7 distribution, LightGBM raw vs LightGBM percentile:
Product                      raw%    pct%   base%
ind_ahor_fin_ult1            0.00   99.57   0.000
ind_aval_fin_ult1            0.00   97.69   0.000
ind_cco_fin_ult1            38.70   12.62   1.056 [modeled]
ind_cder_fin_ult1            0.04   90.44   0.001
ind_cno_fin_ult1            26.73   30.88   0.274 [modeled]
ind_ctju_fin_ult1            2.96   70.97   0.004
ind_ctma_fin_ult1            6.80    6.80   0.056 [modeled]
ind_ctop_fin_ult1           71.63   39.70   0.027
ind_ctpp_fin_ult1           23.34   21.11   0.015
ind_deco_fin_ult1            0.00    8.11   0.000
ind_deme_fin_ult1            0.00    1.71   0.000
ind_dela_fin_ult1            5.76    0.11   0.005
ind_ecue_fin_ult1           50.93   45.46   0.

## Conclusions

### Final Kaggle results from this notebook

| Model | Ranking | Private | Public |
|---|---|---|---|
| CatBoost per-product | raw | **0.02262** | **0.02230** |
| CatBoost per-product | percentile | 0.01877 | 0.01848 |
| LightGBM per-product | percentile | 0.01672 | 0.01642 |
| LightGBM per-product | raw | 0.01568 | 0.01548 |

Reference: LightFM hybrid from the companion notebook scored 0.02491 private, 0.02451 public. Best result from this notebook (CatBoost raw, 0.02262) is 0.00229 below that reference.

### Finding 1 — Raw probability beats percentile on CatBoost by +0.0039

CatBoost raw vs percentile on the private leaderboard: 0.02262 vs 0.01877. The gap is not model quality. It is the ranking function interacting with the popularity fallback.

16 of the 24 products are not modeled. They fall back to `pop_rates[p]` at inference, which sits between 0.0005 and 0.02 depending on the product. Under raw probability the fallback stays at its true base rate and never enters a user's top-7. Under `rank(pct=True)` a constant column maps to percentile ~0.5, the fallback jumps from 0.0005 to 0.5, and it competes head-to-head with weakly modeled products. It leaks into the top-7 of users who will not acquire it, burning slots that should go to `ind_cco`, `ind_recibo`, `ind_nomina`. The distribution check confirmed this: fallback share of top-7 slots rose from ~1% under raw to 10–15% under percentile.

The percentile submission is the same model with a ranking function that breaks the fallback. The mechanism, not the model, costs 0.0039.

### Finding 2 — LightGBM collapsed on the leaderboard despite matching CatBoost in validation

| Model | Validation fullpop | Kaggle private | Drop |
|---|---|---|---|
| CatBoost raw | 0.0236 | 0.02262 | -0.0010 |
| LightGBM raw | 0.0236 | 0.01568 | **-0.0080** |

Same validation score, 8x difference in Kaggle drop. The April to May split has no user-level holdout, so both models see every April user during training. It cannot detect overfitting because it is not out-of-sample.

The original LightGBM configuration:

- `n_estimators=600` fixed, no early stopping
- No depth limit, no `min_child_samples` override, no `reg_lambda`
- `is_unbalance=True` upweights positives by `n_neg / n_pos`, roughly 3000x for products with 0.03% base rate
- Deep unrestricted trees memorize the upweighted training rows

CatBoost avoided this through `early_stopping_rounds=50` on a 20% holdout, `l2_leaf_reg=10`, and `depth=6`. It selected `best_iteration` per product instead of training a fixed 600 trees. The regularization and early stopping are what made the difference in production, not the algorithm choice.

### Finding 2b — Early stopping callbacks did not save LightGBM

A two-stage variant was also tested: 2000 trees with `early_stopping_rounds=50` on a 20% holdout, then retrain on 100% with `best_iteration`. This is the standard fix that "should" have closed the gap, following the same protocol CatBoost uses.

It did not. The callback-limited LightGBM still failed on external validation. Validation performance remained in the same range (~0.0235–0.0240 fullpop) while the Kaggle private score stayed well below CatBoost raw. The two-stage retrain, by itself, did not recover the gap.

The reason is that early stopping only limits the **number** of trees. It does not change what each tree is allowed to learn. With `num_leaves=31`, `min_child_samples=20`, and `reg_lambda=0`, every tree that does get built is still unrestricted and can carve out leaves for a handful of specific users. `is_unbalance=True` compounds this by pushing the model to overfit rare positives. Fewer unrestricted trees is still unrestricted overfitting.

The three regularization parameters that matter — `num_leaves`, `min_child_samples`, `reg_lambda` — were not active in the callback variant. This is the configuration difference that separates a stable model from an unstable one, and it is the reason CatBoost proved stable while LightGBM did not.

**The stability result, stated plainly**: across three configurations tested (600 trees fixed, 2000 trees with callback, callback plus no reg), LightGBM underperformed CatBoost on external validation every time. CatBoost stayed stable across the same April→May→June pipeline. The conclusion is not "LightGBM is worse". It is that **LightGBM's defaults are unstable on this dataset with this class imbalance**, and matching CatBoost requires activating the full regularization stack, not just the early stopping callback.

### Finding 3 — Per-product beats popularity but loses to joint factorization

Best per-product (0.02262) vs popularity baseline (0.01813): **+0.0045**. The feature engineering (demographics, 24 holdings, trajectory, cluster counts) plus masking and the fallback pipeline produce a real signal above trivial ranking.

Best per-product vs LightFM (0.02491): **-0.00229**. The gap is structural. 91% of the target concentrates on 8 products that co-occur strongly. LightFM's embeddings capture product-product co-occurrence across 17 months of transitions. Per-product classifiers see each product in isolation, trained on a single month's snapshot, and cannot represent that "user holds A and B" is a different signal than "user holds A" plus "user holds B".

The per-product approach is not a null result. It clears popularity and clears LightGBM raw. It does not reach the factor model.

### What would have improved this notebook

**Full LightGBM regularization, not just the callback.** The callback variant was tested and failed. The remaining step is to activate `num_leaves=15`, `min_child_samples=200`, `reg_lambda=10` together with the callback. This is the only LightGBM configuration not yet tested, and it is the one most likely to produce 0.020–0.022. Running it would confirm whether the instability is fixable or intrinsic.

**Blend CatBoost raw with LightFM.** The two architectures produce decorrelated errors. A rank-average of the two top-7 lists would probably land around 0.0255 to 0.026, beating both. Not tested because both submissions landed after the Kaggle deadline.

**Skip percentile entirely.** The mechanism was understood before the submission. Running the two percentile files consumed submission slots that could have gone to a regularized LightGBM or a blend.

### Summary

This notebook established three things with concrete numbers:

- **Raw probability is the correct ranking scheme** for per-product classifiers with a popularity fallback, confirmed on CatBoost by +0.0039 on the private leaderboard.
- **Per-product classifiers lose to joint factorization** on this dataset by 0.00229, structurally not by tuning.
- **CatBoost is more stable than LightGBM on this dataset.** Across the configurations tested, including one with early stopping callbacks, LightGBM underperformed CatBoost on external validation every time. CatBoost's regularization stack (`depth=6`, `l2_leaf_reg=10`, `auto_class_weights='Balanced'` on both training stages) provides stability that LightGBM's defaults do not, and the callback alone does not substitute for it.

It also surfaced a validation-design failure. In-sample April to May validation gave both models the same score (~0.0236) but the leaderboard separated them by 0.007. Kaggle is the only out-of-sample measurement in this project.

Best per-product model from this notebook: **CatBoost raw, 0.02262 private, 0.02230 public.**